<a href="https://colab.research.google.com/github/evildead23151/SparseRNN-Systems/blob/main/SparseRNN_Systems_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# =========================================================
# V2 BASELINE SETUP (CPU+GPU SAFE, ZERO-ERROR VERSION)
# =========================================================

import torch
import torch.nn as nn
import numpy as np
import random
import time
import requests

# =========================================================
# 1. ENV SETUP
# =========================================================

def setup_environment(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    print("===== ENVIRONMENT =====")
    print("Device:", device)

    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("⚠️ WARNING: Running on CPU (NO CUDA)")

    return device


# =========================================================
# 2. DATASET
# =========================================================

def load_dataset():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    text = requests.get(url).text

    chars = sorted(list(set(text)))
    vocab_size = len(chars)

    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}

    data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

    print("\n===== DATASET =====")
    print("Length:", len(text))
    print("Vocab size:", vocab_size)

    return data, stoi, itos, vocab_size


# =========================================================
# 3. DATA PREP
# =========================================================

def prepare_data(data, batch_size=64, seq_len=128):
    total_tokens = data.size(0)
    num_batches = total_tokens // (batch_size * seq_len)

    data = data[:num_batches * batch_size * seq_len]
    data = data.view(batch_size, -1)

    print("\n===== DATA SHAPE =====")
    print("Data:", data.shape)

    return data


def get_batch(data, i, seq_len):
    x = data[:, i:i+seq_len]
    y = data[:, i+1:i+seq_len+1]
    return x, y


# =========================================================
# 4. EMBEDDING
# =========================================================

class EmbeddingLayer(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)

    def forward(self, x):
        return self.embedding(x).contiguous()


# =========================================================
# 5. HIDDEN INIT
# =========================================================

def init_hidden(B, H, device):
    h = torch.zeros(B, H, device=device)
    c = torch.zeros(B, H, device=device)
    return (h, c)


# =========================================================
# 6. SAFE BENCHMARK
# =========================================================

def benchmark_model(model_fn, x_input, hidden, device, steps=50, name="model"):
    # Warmup
    for _ in range(10):
        _ = model_fn(x_input, hidden)

    if device.type == "cuda":
        torch.cuda.synchronize()

        starter = torch.cuda.Event(enable_timing=True)
        ender = torch.cuda.Event(enable_timing=True)

        times = []

        for _ in range(steps):
            starter.record()

            _ = model_fn(x_input, hidden)

            ender.record()
            torch.cuda.synchronize()

            times.append(starter.elapsed_time(ender))

        avg = sum(times) / len(times)

    else:
        # CPU timing fallback
        times = []

        for _ in range(steps):
            start = time.time()

            _ = model_fn(x_input, hidden)

            end = time.time()
            times.append((end - start) * 1000)

        avg = sum(times) / len(times)

    print(f"{name}: {avg:.3f} ms")
    return avg


# =========================================================
# 7. MODELS
# =========================================================

class CuDNNLSTM(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)

    def forward(self, x, hidden):
        return self.lstm(x, hidden)


class ManualLSTM(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()

        self.W_x = nn.Linear(hidden_size, 4 * hidden_size, bias=False)
        self.W_h = nn.Linear(hidden_size, 4 * hidden_size, bias=True)

    def forward(self, x, hidden):
        B, T, H = x.shape
        h, c = hidden

        x_proj = self.W_x(x.reshape(B*T, H))
        x_proj = x_proj.view(T, B, 4*H).contiguous()

        outputs = []

        for t in range(T):
            gates = x_proj[t] + self.W_h(h)

            i, f, g, o = torch.chunk(gates, 4, dim=1)

            i = torch.sigmoid(i)
            f = torch.sigmoid(f)
            g = torch.tanh(g)
            o = torch.sigmoid(o)

            c = f * c + i * g
            h = o * torch.tanh(c)

            outputs.append(h)

        return torch.stack(outputs, dim=1), (h, c)


# =========================================================
# 8. MAIN
# =========================================================

if __name__ == "__main__":

    device = setup_environment()

    data, stoi, itos, vocab_size = load_dataset()
    data = prepare_data(data)

    data = data.to(device)

    # CONFIG
    B = 64
    T = 128
    H = 512

    embedding = EmbeddingLayer(vocab_size, H).to(device)

    x_tokens, _ = get_batch(data, 0, T)
    x_embed = embedding(x_tokens)

    hidden = init_hidden(B, H, device)

    cudnn_model = CuDNNLSTM(H).to(device)
    manual_model = ManualLSTM(H).to(device)

    print("\n===== BENCHMARK =====")

    benchmark_model(
        lambda x, h: cudnn_model(x, (h[0].unsqueeze(0), h[1].unsqueeze(0))),
        x_embed,
        hidden,
        device,
        name="cuDNN / CPU fallback"
    )

    benchmark_model(
        manual_model,
        x_embed,
        hidden,
        device,
        name="Manual Dense"
    )

===== ENVIRONMENT =====
Device: cuda
GPU: Tesla T4

===== DATASET =====
Length: 1115394
Vocab size: 65

===== DATA SHAPE =====
Data: torch.Size([64, 17408])

===== BENCHMARK =====
cuDNN / CPU fallback: 12.919 ms
Manual Dense: 23.734 ms


In [18]:
!rm -rf v2
!mkdir v2

In [19]:
%%writefile v2/sparse_kernel.cu

Writing v2/sparse_kernel.cu


In [20]:
%%writefile v2/binding.cpp

Writing v2/binding.cpp


In [21]:
!pip install ninja

In [24]:
!rm -rf v2
!rm -rf /root/.cache/torch_extensions

In [27]:
mkdir v2

In [28]:
%%writefile v2/sparse_kernel.cu

Writing v2/sparse_kernel.cu


In [29]:
%%writefile v2/binding.cpp

Writing v2/binding.cpp


In [35]:
!rm -rf /root/.cache/torch_extensions

In [37]:
import time

name = f"bs_lstm_{int(time.time())}"

with open("v2/binding.cpp", "r") as f:
    code = f.read()

code = code.replace("bs_lstm_PLACEHOLDER", name)

with open("v2/binding.cpp", "w") as f:
    f.write(code)

print("Using module name:", name)

Using module name: bs_lstm_1775697415


In [38]:
!rm -rf /root/.cache/torch_extensions

In [39]:
from torch.utils.cpp_extension import load

sparse_cuda = load(
    name=name,
    sources=["v2/binding.cpp", "v2/sparse_kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O3"],
    extra_cflags=["-O3"]
)

In [40]:
print(sparse_cuda)

<module 'bs_lstm_1775697415' from '/root/.cache/torch_extensions/py312_cu128/bs_lstm_1775697415/bs_lstm_1775697415.so'>


In [57]:
import torch
import torch.nn as nn

class V2BlockSparseLSTM(nn.Module):
    def __init__(self, hidden_size, block_size=64):
        super().__init__()

        self.hidden_size = hidden_size
        self.block_size = block_size
        self.num_blocks = hidden_size // block_size

        self.W = nn.Parameter(
            torch.randn(
                self.num_blocks,
                4,
                block_size,
                block_size,
                device="cuda"
            ) * 0.1
        )

        self.mask = torch.ones(self.num_blocks, dtype=torch.int32, device="cuda")

    def forward(self, x, hidden):
        raise NotImplementedError  # we use CUDA directly for now

In [58]:
model_v2 = V2BlockSparseLSTM(hidden_size=hidden[0].shape[1]).cuda()

In [59]:
h_out, c_out = sparse_cuda.forward(
    x_embed[:, 0].contiguous(),
    hidden[0].contiguous(),
    hidden[1].contiguous(),
    model_v2.W.contiguous(),
    model_v2.mask.contiguous(),
    model_v2.block_size
)

print("h_out:", h_out.shape)
print("c_out:", c_out.shape)

h_out: torch.Size([64, 512])
c_out: torch.Size([64, 512])


In [60]:
import time

name = f"bs_lstm_{int(time.time())}"

# restore placeholder first (important if already replaced earlier)
with open("v2/binding.cpp", "r") as f:
    code = f.read()

# replace ANY previous name with placeholder first (safe reset)
import re
code = re.sub(r'PYBIND11_MODULE\([^,]+,', 'PYBIND11_MODULE(bs_lstm_PLACEHOLDER,', code)

# now inject new name
code = code.replace("bs_lstm_PLACEHOLDER", name)

with open("v2/binding.cpp", "w") as f:
    f.write(code)

print("Using module name:", name)

Using module name: bs_lstm_1775698999


In [61]:
!rm -rf /root/.cache/torch_extensions

In [62]:
from torch.utils.cpp_extension import load

sparse_cuda = load(
    name=name,
    sources=["v2/binding.cpp", "v2/sparse_kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O3"],
    extra_cflags=["-O3"]
)

print("Loaded:", sparse_cuda)

Loaded: <module 'bs_lstm_1775698999' from '/root/.cache/torch_extensions/py312_cu128/bs_lstm_1775698999/bs_lstm_1775698999.so'>


In [63]:
H = hidden[0].shape[1]

model_v2 = V2BlockSparseLSTM(hidden_size=H).cuda()

In [64]:
h_out, c_out = sparse_cuda.forward(
    x_embed[:, 0].contiguous(),
    hidden[0].contiguous(),
    hidden[1].contiguous(),
    model_v2.W.contiguous(),
    model_v2.mask.contiguous(),
    model_v2.block_size
)

print("h_out:", h_out.shape)
print("c_out:", c_out.shape)

h_out: torch.Size([64, 512])
c_out: torch.Size([64, 512])


In [74]:
def run_v3_fused(x_embed, hidden):
    h, c = hidden

    # ⚠️ PASS BLOCK SIZE (THIS IS THE FIX)
    out = sparse_cuda.forward(
        x_embed.contiguous(),
        h.contiguous(),
        c.contiguous(),
        model_v2.W.contiguous(),
        model_v2.mask.contiguous(),
        model_v2.block_size   # ← THIS LINE FIXES EVERYTHING
    )

    # V2 returns (h, c)
    # but since you're looping inside kernel now, treat output properly

    if isinstance(out, (list, tuple)):
        h_new, c_new = out
        return h_new.unsqueeze(1), (h_new, c_new)

    return out, (h, c)

In [75]:
print("\n===== V3 BENCHMARK =====")

benchmark_model(
    run_v3_fused,
    x_embed,
    hidden,
    device,
    name="V3 Fused Sparse"
)


===== V3 BENCHMARK =====
V3 Fused Sparse: 36.182 ms


36.18195358276367

In [78]:
# =========================
# HARD RESET
# =========================
import os, time
os.system("rm -rf v3")
os.system("rm -rf /root/.cache/torch_extensions")

os.makedirs("v3", exist_ok=True)

# =========================
# WRITE CUDA KERNEL
# =========================
with open("v3/sparse_kernel.cu", "w") as f:
    f.write(r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 64

__device__ inline float sigmoidf(float x) {
    return 1.0f / (1.0f + expf(-x));
}

__global__ void fused_lstm_kernel(
    const float* __restrict__ x,
    float* __restrict__ h_out,
    float* __restrict__ h,
    float* __restrict__ c,
    const float* __restrict__ W,
    const int* __restrict__ mask,
    int B, int T, int H, int num_blocks
) {
    int b = blockIdx.x;
    int block_id = blockIdx.y;
    int tid = threadIdx.x;

    if (b >= B || block_id >= num_blocks) return;
    if (mask[block_id] == 0) return;

    int h_offset = block_id * BLOCK_SIZE + tid;

    __shared__ float W_shared[4][BLOCK_SIZE][BLOCK_SIZE];

    for (int g = 0; g < 4; g++) {
        for (int i = tid; i < BLOCK_SIZE * BLOCK_SIZE; i += BLOCK_SIZE) {
            int r = i / BLOCK_SIZE;
            int c_idx = i % BLOCK_SIZE;
            W_shared[g][r][c_idx] =
                W[block_id * 4 * BLOCK_SIZE * BLOCK_SIZE +
                  g * BLOCK_SIZE * BLOCK_SIZE +
                  r * BLOCK_SIZE + c_idx];
        }
    }
    __syncthreads();

    float h_val = h[b * H + h_offset];
    float c_val = c[b * H + h_offset];

    float h_reg[BLOCK_SIZE];
    #pragma unroll
    for (int k = 0; k < BLOCK_SIZE; k++) {
        h_reg[k] = h[b * H + block_id * BLOCK_SIZE + k];
    }

    for (int t = 0; t < T; t++) {

        float x_val = x[t * B * H + b * H + h_offset];

        float i_val = 0.f, f_val = 0.f, g_val = 0.f, o_val = 0.f;

        #pragma unroll
        for (int k = 0; k < BLOCK_SIZE; k++) {
            float h_k = h_reg[k];

            i_val += W_shared[0][tid][k] * h_k;
            f_val += W_shared[1][tid][k] * h_k;
            g_val += W_shared[2][tid][k] * h_k;
            o_val += W_shared[3][tid][k] * h_k;
        }

        i_val += x_val;
        f_val += x_val;
        g_val += x_val;
        o_val += x_val;

        i_val = sigmoidf(i_val);
        f_val = sigmoidf(f_val);
        g_val = tanhf(g_val);
        o_val = sigmoidf(o_val);

        c_val = f_val * c_val + i_val * g_val;
        h_val = o_val * tanhf(c_val);

        h_reg[tid] = h_val;

        h_out[t * B * H + b * H + h_offset] = h_val;
    }

    h[b * H + h_offset] = h_val;
    c[b * H + h_offset] = c_val;
}

void launch_fused_lstm(
    torch::Tensor x,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor W,
    torch::Tensor mask,
    torch::Tensor h_out
) {
    int B = x.size(0);
    int T = x.size(1);
    int H = x.size(2);
    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    fused_lstm_kernel<<<grid, block>>>(
        x.data_ptr<float>(),
        h_out.data_ptr<float>(),
        h.data_ptr<float>(),
        c.data_ptr<float>(),
        W.data_ptr<float>(),
        mask.data_ptr<int>(),
        B, T, H, num_blocks
    );
}
''')

# =========================
# WRITE BINDING
# =========================
with open("v3/binding.cpp", "w") as f:
    f.write(r'''
#include <torch/extension.h>

void launch_fused_lstm(
    torch::Tensor x,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor W,
    torch::Tensor mask,
    torch::Tensor h_out
);

torch::Tensor forward(
    torch::Tensor x,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor W,
    torch::Tensor mask
) {
    auto B = x.size(0);
    auto T = x.size(1);
    auto H = x.size(2);

    auto h_out = torch::zeros({B, T, H}, x.options());

    launch_fused_lstm(x, h, c, W, mask, h_out);

    return h_out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward, "Fused Sparse LSTM Forward");
}
''')

# =========================
# COMPILE
# =========================
from torch.utils.cpp_extension import load

name = f"bs_lstm_v3_{int(time.time())}"

sparse_cuda = load(
    name=name,
    sources=["v3/binding.cpp", "v3/sparse_kernel.cu"],
    verbose=False,
    extra_cuda_cflags=["-O3"],
    extra_cflags=["-O3"]
)

print("Loaded:", name)

# =========================
# RUN FUNCTION
# =========================
def run_v3_fused(x_embed, hidden):
    h, c = hidden

    out = sparse_cuda.forward(
        x_embed.contiguous(),
        h.contiguous(),
        c.contiguous(),
        model_v2.W.contiguous(),
        model_v2.mask.contiguous()
    )

    return out, (h, c)

# =========================
# BENCHMARK
# =========================
print("\n===== V3 BENCHMARK =====")

benchmark_model(
    run_v3_fused,
    x_embed,
    hidden,
    device,
    name="V3 TRUE FUSED"
)

RuntimeError: Error building extension 'bs_lstm_v3_1775700930': [1/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output sparse_kernel.cuda.o.d -DTORCH_EXTENSION_NAME=bs_lstm_v3_1775700930 -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.12 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr -gencode=arch=compute_75,code=compute_75 -gencode=arch=compute_75,code=sm_75 --compiler-options '-fPIC' -O3 -std=c++17 -c /content/v3/sparse_kernel.cu -o sparse_kernel.cuda.o 
FAILED: [code=2] sparse_kernel.cuda.o 
/usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output sparse_kernel.cuda.o.d -DTORCH_EXTENSION_NAME=bs_lstm_v3_1775700930 -DTORCH_API_INCLUD